# SentryICU — Module 1: Baseline Clinical Risk MLP, Activations & Optimizers
### Cross-Hospital Generalizable Early Warning System for ICU Deterioration

**Module 1 Objectives:**
1. Ingest real-world ICU patient records from PhysioNet 2019 Challenge (Hospital A cohort).
2. Extract standardized 73-dimensional clinical feature vectors across vitals, labs, and demographics.
3. Build a modular PyTorch Multi-Layer Perceptron (`ClinicalRiskMLP`).
4. Handle severe class imbalance with positive-weighted Binary Cross-Entropy (`pos_weight`).
5. Perform comprehensive empirical ablations across 4 activation functions (`ReLU`, `LeakyReLU`, `GELU`, `ELU`) and 4 optimizers (`Adam`, `AdamW`, `SGD`, `RMSprop`).
6. Log training dynamics to TensorBoard and export the top-performing checkpoint (`models/module1_best_mlp.pt`).

In [ ]:
# Step 0: Environment Setup, Drive Mounting & GPU Verification
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
PROJECT_DIR = '/content/drive/MyDrive/SentryICU'
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/runs', exist_ok=True)
os.chdir(PROJECT_DIR)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Working directory: {os.getcwd()}")
print(f"[+] PyTorch device: {device} (CUDA Available: {torch.cuda.is_available()})")

!pip install -q scikit-learn pandas numpy matplotlib seaborn tabulate tensorboard
%load_ext tensorboard

In [ ]:
# Step 1: Real Clinical Feature Extraction Pipeline (PhysioNet 2019 Hospital A)
import pandas as pd
import numpy as np
import glob
from tqdm.auto import tqdm

ACTUAL_DATA_DIR = '/content/drive/MyDrive/SentryICU/data/raw/training_setA/training'
if not os.path.exists(ACTUAL_DATA_DIR):
    ACTUAL_DATA_DIR = '/content/drive/MyDrive/SentryICU/data/raw/training_setA'

KEY_VITALS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
KEY_LABS = ['Lactate', 'WBC', 'Creatinine', 'BUN', 'Glucose', 'Platelets', 'Hgb']

def extract_patient_features(psv_path):
    try:
        df = pd.read_csv(psv_path, sep='|')
        if len(df) == 0: return None
        label = int(df['SepsisLabel'].max())
        features = {}
        for col in KEY_VITALS + KEY_LABS:
            series = df[col].ffill().bfill()
            if series.isna().all():
                features[f'{col}_mean'] = 0.0; features[f'{col}_min'] = 0.0; features[f'{col}_max'] = 0.0
                features[f'{col}_std'] = 0.0; features[f'{col}_last'] = 0.0
            else:
                features[f'{col}_mean'] = float(series.mean())
                features[f'{col}_min'] = float(series.min())
                features[f'{col}_max'] = float(series.max())
                features[f'{col}_std'] = float(series.std()) if len(series) > 1 else 0.0
                features[f'{col}_last'] = float(series.iloc[-1])
        features['Age'] = float(df['Age'].iloc[0]) if not pd.isna(df['Age'].iloc[0]) else 60.0
        features['Gender'] = float(df['Gender'].iloc[0]) if not pd.isna(df['Gender'].iloc[0]) else 0.0
        features['Max_ICULOS'] = float(df['ICULOS'].max())
        features['Label'] = label
        return features
    except Exception:
        return None

files = glob.glob(os.path.join(ACTUAL_DATA_DIR, '*.psv'))[:2500]
print(f"[*] Processing {len(files)} patient files from {ACTUAL_DATA_DIR}...")
rows = [extract_patient_features(f) for f in tqdm(files)]
cohort_df = pd.DataFrame([r for r in rows if r is not None]).fillna(0)
cohort_df.to_csv(f'{PROJECT_DIR}/data/processed_cohort_A.csv', index=False)
print(f"[+] Cohort built: {cohort_df.shape[0]} patients, {cohort_df.shape[1]-1} features | Prevalence: {cohort_df['Label'].mean():.2%}")

In [ ]:
# Step 2: PyTorch Dataset, Standard Scaling & Class Imbalance Weighting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

feature_cols = [c for c in cohort_df.columns if c != 'Label']
X_raw = cohort_df[feature_cols].values
y_raw = cohort_df['Label'].values

X_train, X_val, y_train, y_val = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

class ICUDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_dataset = ICUDataset(X_train_scaled, y_train)
val_dataset = ICUDataset(X_val_scaled, y_val)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
pos_weight_val = num_neg / max(num_pos, 1)
pos_weight = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
print(f"[*] Calculated Class Imbalance Pos-Weight: {pos_weight_val:.2f}")

In [ ]:
# Step 3: Modular MLP & Full Ablation Matrix (Activations & Optimizers)
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import roc_auc_score, average_precision_score
from tabulate import tabulate

class ClinicalRiskMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], activation='relu', dropout_rate=0.2):
        super(ClinicalRiskMLP, self).__init__()
        act_map = {
            'relu': nn.ReLU(),
            'leaky_relu': nn.LeakyReLU(negative_slope=0.1),
            'gelu': nn.GELU(),
            'elu': nn.ELU(alpha=1.0)
        }
        act_layer = act_map.get(activation.lower(), nn.ReLU())
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(act_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x)

def train_and_evaluate(activation='relu', optimizer_name='adam', lr=1e-3, epochs=20):
    exp_name = f"Act_{activation}__Opt_{optimizer_name}"
    writer = SummaryWriter(log_dir=f"{PROJECT_DIR}/runs/{exp_name}")
    model = ClinicalRiskMLP(input_dim=len(feature_cols), activation=activation).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt_name = optimizer_name.lower()
    if opt_name == 'adam': optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    elif opt_name == 'adamw': optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    elif opt_name == 'sgd': optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    elif opt_name == 'rmsprop': optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=1e-4)
    else: optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'val_auroc': [], 'val_auprc': []}
    for epoch in range(1, epochs + 1):
        model.train(); running_train_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(X_b)
            loss = criterion(logits, y_b)
            loss.backward(); optimizer.step()
            running_train_loss += loss.item() * len(y_b)
        train_loss = running_train_loss / len(train_dataset)

        model.eval(); running_val_loss = 0.0; val_preds, val_targets = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits = model(X_b)
                loss = criterion(logits, y_b)
                running_val_loss += loss.item() * len(y_b)
                probs = torch.sigmoid(logits).cpu().numpy()
                val_preds.extend(probs); val_targets.extend(y_b.cpu().numpy())
        val_loss = running_val_loss / len(val_dataset)
        val_preds = np.array(val_preds).flatten(); val_targets = np.array(val_targets).flatten()
        val_auroc = roc_auc_score(val_targets, val_preds) if len(np.unique(val_targets)) > 1 else 0.5
        val_auprc = average_precision_score(val_targets, val_preds) if len(np.unique(val_targets)) > 1 else 0.0
        history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
        history['val_auroc'].append(val_auroc); history['val_auprc'].append(val_auprc)
        writer.add_scalar('Loss/Train', train_loss, epoch); writer.add_scalar('Loss/Val', val_loss, epoch)
        writer.add_scalar('Metrics/Val_AUROC', val_auroc, epoch); writer.add_scalar('Metrics/Val_AUPRC', val_auprc, epoch)
    writer.close()
    return model, history, val_preds, val_targets

activations = ['relu', 'leaky_relu', 'gelu', 'elu']
optimizers = ['adam', 'adamw', 'sgd', 'rmsprop']
results = []; models_dict = {}; eval_dict = {}

for act in activations:
    print(f"[*] Training: Activation = {act.upper():<10} | Optimizer = AdamW")
    m, h, p, t = train_and_evaluate(activation=act, optimizer_name='adamw')
    key = f"Act_{act}__Opt_adamw"; models_dict[key] = m; eval_dict[key] = (p, t, h)
    results.append([f"Activation: {act.upper()}", "AdamW", f"{h['train_loss'][-1]:.4f}", f"{h['val_loss'][-1]:.4f}", f"{h['val_auroc'][-1]:.4f}", f"{h['val_auprc'][-1]:.4f}"])

for opt in optimizers:
    if opt == 'adamw': continue
    print(f"[*] Training: Activation = GELU       | Optimizer = {opt.upper()}")
    m, h, p, t = train_and_evaluate(activation='gelu', optimizer_name=opt)
    key = f"Act_gelu__Opt_{opt}"; models_dict[key] = m; eval_dict[key] = (p, t, h)
    results.append(["Activation: GELU", opt.upper(), f"{h['train_loss'][-1]:.4f}", f"{h['val_loss'][-1]:.4f}", f"{h['val_auroc'][-1]:.4f}", f"{h['val_auprc'][-1]:.4f}"])

print("\n" + tabulate(results, headers=["Experiment", "Optimizer", "Train Loss", "Val Loss", "Val AUROC", "Val AUPRC"], tablefmt="fancy_grid"))

In [ ]:
# Step 4: Publication Visualizations & Model Checkpointing
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for k, (p, t, h) in eval_dict.items(): axes[0].plot(h['val_loss'], label=k.replace('__', ' | '))
axes[0].set_title("Validation Loss Comparison", fontweight='bold'); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(fontsize=7); axes[0].grid(True, linestyle='--', alpha=0.5)

for k, (p, t, h) in eval_dict.items():
    fpr, tpr, _ = roc_curve(t, p); auc = roc_auc_score(t, p)
    axes[1].plot(fpr, tpr, label=f"{k} (AUC={auc:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4); axes[1].set_title("ROC Curves (ICU Sepsis Detection)", fontweight='bold'); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].legend(fontsize=7); axes[1].grid(True, linestyle='--', alpha=0.5)

for k, (p, t, h) in eval_dict.items():
    prec, rec, _ = precision_recall_curve(t, p); pr_auc = average_precision_score(t, p)
    axes[2].plot(rec, prec, label=f"{k} (PR-AUC={pr_auc:.3f})")
axes[2].set_title("Precision-Recall Curves", fontweight='bold'); axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision"); axes[2].legend(fontsize=7); axes[2].grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

best_key = "Act_gelu__Opt_adamw"
torch.save({
    'model_state_dict': models_dict[best_key].state_dict(),
    'feature_cols': feature_cols,
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
    'activation': 'gelu'
}, f'{PROJECT_DIR}/models/module1_best_mlp.pt')
print(f"[+] Saved top checkpoint to {PROJECT_DIR}/models/module1_best_mlp.pt")

%tensorboard --logdir /content/drive/MyDrive/SentryICU/runs